In [ ]:
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, precision_score, accuracy_score

# Misc tools
from tqdm.notebook import tqdm
import os
import inspect
import warnings
warnings.filterwarnings("ignore")

# NB5 Initial Modeling Considerations

## Introduction

The goal of this notebook is to determine the appropriate models and hyper-parameters to implement into a grid search used to identify the best model for classifying the citations into publication tiers. In the interest of making the grid search as computationally efficient as possible, we will first evaluate the potential *score* of each potential classification model based on their performance on pseudo-optimized parameters and use those evaluation finds to determine which models and range of hyper-parameter to include in the grid-search, and which to leave out.

### Defining and Evaluating a Model's *Score*

#### On Precision vs Accuracy
Here, we define the model's score as the model's accuracy score averaged with the model's weighted precision score where a higher average corresponds to a higher score. This custom score is deployed using function `custom_score`. This custom score was chosen so that the precision of the model would be equally weighed against its accuracy. As a result, we will have a model that correctly classifies the most citations (accuracy), while also ensuring that achieving the most number of correctly classified citations is not done at the expense of citations belonging to the minority classes being wrongfully classified (precision). 

#### On Over and Under Fitting
In addition to the precision and accuracy considerations of the model, we will also evaluate the model on its fitting to the training and validation set. Specifically, we will use cross-validation to 


Since the number of rows in our data set in relatively small (< 20,000), we will use cross-validation as the method to generate validation sets, as well as 



The accuracy will be 

will deploy a custom function that will 

we will examine each model using the following protocol:

Goal: Identify the ideal models and hyper-parameter optimization ranges to implement into gridsearchCV to not waste computing time on testing every model with every range of parameter optimization values.

1. Fit the model to our data once using default hyper-parameter settings to gauge a baseline training and testing accuracy.
2. 
3. Investigate possible sources of error 
3. Perform a raster-scan fitting of a large range of hyper-parameters to determine an 

In [ ]:
ml_df = pd.read_pickle('2023-04-04_NB4_ml_df.pkl')

`ml_df` is our data set for performing machine learning. Here the target column is `PubTier`, which is the tier that each citation is classified in, while the remaining statistics for the first, second and third authors are the features.

In [ ]:
ml_df.info()

In [ ]:
# Define features and target colums
X = ml_df.drop('PubTier', axis = 1)
y = ml_df['PubTier']

# Split the data set
X_remainder, X_test, y_remainder, y_test =  train_test_split(X, y, test_size=0.25, random_state = 1)

In [ ]:
def plt_hp_opt(x, train_scores, test_scores, xlabel, title):
    '''
    
    This function is designed to create a matplotlib plt isntance for plotting the results of hyperparameter optimizations.
    It is designed so any additional methods to edit the plot can be done after this function is called.
    
    Parameters:
    ----------
    x: The values for the x-axis
    train_scores: A list of values of the training scores
    test_scores: A list of values of the test scores
    xlabel: A str to be used as the xlabel
    title: A str to be added to the title
    
    '''
    plt.plot(x, train_scores, marker = '.', label = 'Train')
    plt.plot(x, test_scores, marker = '.', label = 'Test')
    plt.xlabel(xlabel)
    
    y_ticks = np.arange(0.6, 1.02, 0.02)
    y_labels = ['{:.0%}'.format(i) for i in y_ticks]
    plt.yticks(y_ticks, y_labels)
    
    plt.ylabel('Model Accuracy')
    plt.title(f'{title} Hyperparameter Optimization')
    plt.legend()
    plt.grid()

In [ ]:
# Define a custom scoring function that combines accuracy and precision scores

def custom_score_func(y_val, y_pred):
    '''
    
    This function finds the average of the accuracy score and precision scores to use to find the best
    fitting model in GridSearchCV.
    
    Parameters:
    ---------
    y_val: Validation set of the input target data, used as the true values.
    y_pred: Predicted labels returned by the classifer.
    
    Return:
    ------
    ret: The accuracy and precision score averaged.
    
    '''
    acc = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, average='weighted')
    return (acc + precision) / 2

custom_score = make_scorer(custom_score_func, greater_is_better = True)

In [ ]:
def update_params(func, params):
    
    assert isinstance(params, dict), 'Input params must be a dictionary'
    
    bound_args = inspect.signature(func).bind(**params)  # bind the arguments to the function parameters
    bound_args.apply_defaults()  # use default values for any unspecified parameters
    
    return func(*bound_args.args, **bound_args.kwargs)  # call the function with the updated parameters

def get_scores(model_obj, decomp_obj = None):

    # Make sure the model_obj is a classifcation model_obj by checking for the presence of a method
    assert hasattr(model_obj, 'predict_proba'), "Input model_obj is not a classification model_obj"
   
    # Inititate Dictionary
    dic = {'Accuracy Score':{}, 'Weighted Precision Score': {}, 'Custom Score': {}}
    
    #we give our estimators as a list of tuples
    
    if decomp_obj != None:
        
        # Make sure the decomp_obj is a classifcation model_obj by checking for the presence of a method
        assert isinstance(decomp_obj, PCA), "Input decomp_obj is not a PCA obj"
        
        estimators = [('normalise', StandardScaler()), # scalar
                      ('scalar', decomp_obj), # dimentionality reduction 
                      ('model_obj', model_obj)] # fit to model_obj
    elif decomp_obj == None:
        estimators = [('normalise', StandardScaler()), # scalar
                      ('model_obj', model_obj)] # fit to model_obj
        
    # Initiate Pipeline
    pipe = Pipeline(estimators)

    # Fit to model_obj
    pipe.fit(X_remainder, y_remainder)

    # Do Cross-Validation for 5 folds and return the scores

    # Accuracy Scores
    cvs_acc = cross_validate(pipe, X_remainder, y_remainder, cv = 5, return_train_score=True) 

    # Precision Scores
    cvs_pre = cross_validate(pipe, X_remainder, y_remainder, cv = 5, scoring='precision_weighted', return_train_score=True)

    # Custom Scores
    cvs_cust = cross_validate(pipe, X_remainder, y_remainder, cv = 5, scoring=custom_score, return_train_score=True)

    # Add values to dictionary
    dic['Accuracy Score']['Training'] = np.mean(cvs_acc['train_score'])
    dic['Accuracy Score']['Validation'] = np.mean(cvs_acc['test_score'])
    dic['Custom Score']['Training'] = np.mean(cvs_cust['train_score'])
    dic['Custom Score']['Validation'] = np.mean(cvs_cust['test_score'])
    dic['Weighted Precision Score']['Training'] = np.mean(cvs_pre['train_score'])
    dic['Weighted Precision Score']['Validation'] = np.mean(cvs_pre['test_score'])
    
    return dic

def decomp_fit(decomp_obj):

    # Make sure the decomp_obj is a classifcation model_obj by checking for the presence of a method
    assert isinstance(my_PCA, PCA), "Input decomp_obj is not a PCA obj"

    estimators = [('normalise', StandardScaler()), # scalar
                  ('scalar', decomp_obj)] # dimentionality reduction
    
    # Initiate Pipeline
    pipe = Pipeline(estimators)
    
    # Fit the pipeline
    pipe.fit(X_remainder, y_remainder)

    # Use 5-fold cross-validation to evaluate the pipeline's performance
    scores = cross_val_score(pipe, X_remainder, y_remainder, cv=5)

    return pipe.named_steps['scalar']

    
    

In [ ]:
color_pal = sns.color_palette("Set2")

class score_models:
    
    def __init__(self, model_obj, model_param, decomp_obj = None, model_name = ''):
        self.model_name = model_name
        self.model_param = model_param

        # Make sure the model_obj is a classifcation model_obj by checking for the presence of a method
        assert hasattr(model_obj, 'predict_proba'), "Input model_obj is not a classification model_obj"
        
        if decomp_obj != None:
            # Make sure the decomp_obj is a classifcation model_obj by checking for the presence of a method
            assert isinstance(decomp_obj, PCA), "Input decomp_obj is not a PCA obj"
        
        self.model_obj = model_obj
        self.decomp_obj = decomp_obj
        self.model_inst = update_params(self.model_obj, self.model_param)
        self.score_dic = get_scores(self.model_inst, self.decomp_obj)
        self.search_dic = None
    
    def set_decomp_obj(self, decomp_obj):
            
        assert isinstance(decomp_obj, PCA), "Input decomp_obj is not a PCA obj"
        
        self.decomp_obj = decomp_obj

    def plt_scores(self):
        
        assert self.score_dic != None, 'Instance Must be Scored Before Plotted'
        
        plt.ioff() # figures will not automatically be shown
        dic = self.score_dic
        fig = plt.figure()
        ax = fig.subplots(1,1)
        
        j = 0
        for key in list(dic.keys()):
            for sub_key in list(dic[key]):
                bar_height = dic[key][sub_key]
                
                if sub_key == 'Training':
                    c = color_pal[0]
                elif sub_key == 'Validation':
                    c = color_pal[1]
                
                ax.bar(j, bar_height, width = 0.25 , label = sub_key, color = c)
                text = '{:0.2%}'.format(bar_height)
                ax.text(j, bar_height + 0.0005, text, fontsize = 'small', horizontalalignment='center', verticalalignment='bottom') 
                
                j += 0.35
            j += 0.5
        
        ax.set_ylim([0.6, 0.85])
        ax.set_ylabel('Model Accuracy')
        ax.set_title(f'Results of {self.model_name} Default Parameter Fit')
        h, l = ax.get_legend_handles_labels()
        ax.legend(h[:2], l[:2], title = 'Set Type')
        x_ticks = np.arange(0.175, 3.175, 1.2)
        ax.set_xticks(x_ticks, list(dic.keys()))
        ax.set_yticks([],[])
        
        return fig
    
    
    def plt_hp_opt(self):
        
        '''

        This function is designed to create a matplotlib plt isntance for plotting the results of hyperparameter optimizations.
        It is designed so any additional methods to edit the plot can be done after this function is called.

        Parameters:
        ----------


        '''
        
        assert self.search_dic != None, 'Parameter Search Must be Carried Before Plotting'
        
        dic = self.search_dic
        fig = plt.figure()
        ax = fig.subplots(1,1)
        ax.grid()

        # Name of the parameter being optimized
        param_name = dic['ParameterName']

        # Values for the x-axis
        x = dic['ParameterValues']

        # Plotting Average Accuracy Scores
        for key in list(dic.keys())[:2]:
            y = dic[key]
            ax.plot(x, y, marker = '.', label = key)

        ax.set_xlabel(f'Values of {param_name}')

        y_ticks = np.arange(0.6, 1.02, 0.02)
        y_labels = ['{:.0%}'.format(i) for i in y_ticks]
        ax.set_yticks(y_ticks, y_labels)

        ax.set_ylabel('Model Accuracy')
        ax.set_ylim([0.6, 0.8])
        ax.set_title(f'{self.model_name} Hyperparameter Optimization')
        ax.legend();
        
        return fig
        
            
    def search(self, param_key, param_arr):

        assert isinstance(param_key, str), 'Parameter Key Must be String'
        
        # Initiating the dictionary to store results
        dic = {'Training Score':[], 'Validation Score': [], 'ParameterValues': param_arr, 'ParameterName': str}
        
        # Storing the name of the optimized parameter
        dic['ParameterName'] = param_key
        
        # Looping over parameter values
        for param_val in tqdm(param_arr):
            
            # Create a copy of the model_param dictionary 
            param_dic = self.model_param.copy()
            
            # replacing parameter dictionary with updated value
            param_dic[param_key] = param_val
            
            # Defining the new model using the updated parameters
            new_model = update_params(self.model_obj, param_dic)
            
            # scoring the new model
            self.score_dic = get_scores(new_model, self.decomp_obj)

            dic['Training Score'].append(self.score_dic['Accuracy Score']['Training'])
            dic['Validation Score'].append(self.score_dic['Accuracy Score']['Validation'])

        self.search_dic = dic

## Model 1: Logistic Regression

### Initial Model Fitting
Let's fit a logistic regression model to our data once using default hyper-parameter settings to gauge a baseline training and testing accuracy and precision metrics. We will use a train-test split of `test_size` = 25%, set `solver` to `lbfgs` to specify the algorithm to when fitting the logistic regression model, and set `random_state`= 1 to ensure that the model can be replicated with the same results every time it is trained. The training and testing accuracy scores will be outputted.

In [ ]:
model_obj = LogisticRegression
sm_logreg = score_models(model_obj, model_param = {'solver':'lbfgs', 'random_state': 1, 'C':1}, model_name = 'LogReg')

In [ ]:
sm_logreg.plt_scores();
plt.show()

With the default hyper-parameter settings (`C`= 1) the logistic regression model was able to fit the training data with 76.12% accuracy and the testing data with 75.60% accuracy, additionally, the difference between training and testing scores being < 1% suggests that there is minimal over-fitting which could hopefully be solved optimizing the weight of the cost function in for the assigned penalty (L2, in our case). The weighted precision score of 75.83% suggests that observations were classified into their correct class nearly 76% of the time, though there is much room for improvement.

### Optimizing Model Accuracy

#### Part 1: Addressing Multicollinearity
Since logistic regression is a linear classification model, before perform considering the landscape of hyper-parameters to increase model accuracy, we should first access the other potential issues that could hinder the accuracy of the model, such as multicollinearity.

We can investigate multicollinearities between features using a heat map.

In [ ]:
ml_corr = ml_df.drop('PubTier', axis = 1).corr()

In [ ]:
sns.heatmap(np.abs(ml_corr), vmin = 0, vmax = 1)
plt.title('Absolute Value of Correlations Between Features')

From the heatmap, we see that there are areas where collinearity between features is strong, i.e., > 0.5. Let's rescale the heatmap to highlight those collinearities.

In [ ]:
sns.heatmap(np.abs(ml_corr), vmin = 0.5, vmax = 1)
plt.title('Absolute Value of Correlations Between Features\nRescaled to Feature Correlations >= 0.5')

From this rescaled heat map, we see that most of the features have a correlation to another feature that has an absolute value over 0.5, with some features have correlations to another feature with a value over 0.8. Since our data set has < 50 features and most of the features attributes to the multicollinearity, removing select features through feature selection is unlikely going to solve multicollinearity issue. Instead, removing what limited features the model does have could result in the model being less generalizable if the limited columns selected do not adequately capture the variance in the data set. Overall, given the wide-spread multicollinearity and limited number of features, pursuing logistic regression as our classification model may not be ideal. 

However, we could employ a method of dimensionality reduction, such as principal vector component analysis (PCA) to reduce the number of columns, which could reduce multicollinearity, while preserving the variance in the data set. The effectiveness of PCA at capturing the variance of the data set can be quantified using the *explained variance ratio* for each of principle component (PC) describe the variation in the data set. Specifically, the explained variance ratio describes the *percentage* of the variance that each PC is able to account for, with the goal of increasing the number of PCs until all the variance in the data set can be adequately captured.

We can identify the number of PCs that adequately capture the variance in our data set by making a line plot of the explained variance ratios and identifying the elbow in the plot to decide how many PCs are  adequate. The number of PCs at the elbow in the plot will identify the threshold for where any additional PCs will not substantially raise the explained variance, and are therefore redundant at explaining the variance in the data set.

The number of PCs can be controlled using the `n_components` hyper-parameter. Let's examine how the number of principal components by examining the *explained variance ratios* for each.

In [ ]:
n_list = np.arange(2, 44, 4)
EVR_list = []

for n in tqdm(n_list):
    # Initiate PCA Model
    my_PCA = PCA(n_components = n)
    
    # Fit to data set
    fit_obj = decomp_fit(my_PCA)
    
    EVR = fit_obj.explained_variance_ratio_.mean()
    EVR_list.append(EVR)

In [ ]:
plt.plot(n_list, EVR_list, marker = '.')
plt.xlabel('Number of Principal Components')
plt.ylabel('Proportion of Variance Explained')
plt.grid()
x_ticks = np.arange(2, 44, 4)
plt.xticks(x_ticks, x_ticks);

Here we see that the elbow in the plot is between 6-10 PCs suggesting that any additional PCs added beyond that will not substantially raise the explained variance. However, it appears that the explained variance ratio does not truly become constant until beyond 30 PCs so that the ideal number of PCs is anywhere between 10-30. This may be worth exploring during a grid search, but first we need to address if reducing the number of features using PCA will address the issue with multicollinearity. Let's look at a heat map of the PCA components for `n_components` = 10 and `n_components` = 30 to investigate.

In [ ]:
for i in [10, 30]:
    # Initiate PCA Model
    my_PCA = PCA(n_components = i)
    
    # Fit to data set
    fit_obj = decomp_fit(my_PCA)

    # Converting to DF to get correlation matrix. Transposing the array so that components are the columns.
    pca_corr = pd.DataFrame(my_PCA.components_.T).corr()

    sns.heatmap(np.abs(pca_corr), vmin = 0, vmax = 1)
    plt.title(f'Absolute Value of Correlations Between PCA Components\nn_components = {i}')
    plt.show()

From these results we observe that there are no longer any columns that have a correlation of > 0.5 for both the lower bound of `n_components` = 10 and upper bound of `n_components` = 30. These results suggests that any number of components between 6 and 30 selected by the grid search would not result in features that have strong correlations in between them, therefore suggesting the logistic regression could still be a feasible model for the data set when PCA in employed.

#### Part 2: Optimizing Hyper Parameters

We can optimize the hyper-parameter `C` in the logistic regression model in order to optimize the weight of the cost function. Let's examine how the training and testing accuracy changes for values of `C` over a large range ($10^{-5},10^{5}$).

In [ ]:
c_vals = np.asarray([10**i for i in range(-5, 5)])
sm_logreg.search('C', c_vals)

In [ ]:
fig = sm_logreg.plt_hp_opt();
ax, = fig.get_axes();
ax.set_xscale('log');
plt.show()

From these results, we see that logistic regression model suffers from *under fitting* for `C` $< 10^{-2}$ and switches to suffering from over fitting from `C` $> 10^{-2}$. It appears that the least amount of over and under fitting occurs at `C` $= 10^{-2}$, so a grid search for the optimal C values could be around `C` $= 10^{-2}$ (range of 0.005 to 0.0105, for example).

To further isolate the optimal range of `C` values for grid searching, let's apply the same analysis for the `C` values for the data set reduced using PCA with the lower bound `n_components` = 6 and upper bound `n_components` = 30.

In [ ]:
c_vals = np.asarray([10**i for i in range(-5, 5)])

j = 0
for i in [10, 30]:
    # Initiate PCA Model
    my_PCA = PCA(n_components = i)
    
    # Reset Decomp obj
    sm_logreg.set_decomp_obj(my_PCA)
    
    # Perform Optimization
    sm_logreg.search('C', c_vals)
    
    fig = sm_logreg.plt_hp_opt();
    ax, = fig.get_axes();
    ax.set_xscale('log');
    ax.set_title(f'n_components = {i}\n LogReg Hyperparameter Optimization');
    plt.show()
    j += 1

We get similar results the data set reduced using PCA with the lower bound `n_components` = 6 and upper bound `n_components` = 30 as we did the original data set, such as the logistic regression model suffers from *under fitting* for `C` $< 10^{-2}$ and switches to suffering from over fitting from `C` $> 10^{-2}$. However, for the data set reduced using `n_components` = 6, the under fitting for `C` $< 10^{-2}$  is much more exaggerated at `C` $<= 10^{-1}$ than the other two examples. Overall, this finding further supports the range of values for `C` for grid search to consider to be closer to $10^{-2}$ than $10^{-3}$ and $10^{-1}$ to optimize model fitting.

---

## Model 2: KNN Classifers

### Initial Fitting
Let's fit a K nearest-neighbor (KNN) model to our data once using default hyper-parameter settings to gauge a baseline training and testing accuracy and precision metrics. We will use the same train-test split with `test_size` = 25% as was used for the logistic regression model, as well as the feature columns scaled using `StandardScalar`. 

In [ ]:
model_obj = KNeighborsClassifier
sm_knn = score_models(model_obj, model_param = {'n_neighbors':5}, model_name = 'KNN')

In [ ]:
sm_knn.plt_scores();
plt.show()

With the default hyper-parameter settings (`n_neighbors`= 5) the KNN model was able to fit the training data with 78.06% accuracy and the testing data with 66.92% accuracy. Unlike the logistic regression model, the difference between training and testing scores was over > 1%. In fact, the difference was > 10% (11.14%) suggesting there is most over fitting in the KNN model with the default parameters than the logistic regression model. The weighted precision score of 68.00% suggests that observations were classified into their correct class nearly 68% of the time, which is also a lower percentage of precision compared to that of the logistic regression model (78%). These results conclude that logistic regression may be the ideal model for classifying the citations into publication tiers, however let's test to see if changing the value of `n_neighbors` could increase the KNN model accuracy.

### Optimizing Model Accuracy

Let's see the model accuracy for KNN changes with integer step-size values of `n_neighbors` over a range of 5 to 20.

In [ ]:
nn_list = np.arange(5, 43, 2)
sm_knn.search('n_neighbors', nn_list)

In [ ]:
fig = sm_knn.plt_hp_opt();
plt.show()

From the results, we observe the training and testing accuracy converging towards 0.72 as `n_neighbors` increases. Although past `n_neighbors` = 35, the testing accuracy starts to diverge from 0.72, which would be consistent with the model having a greater tendency to over fit when the `n_neighbors` reaching the number of features in the data set (44). Overall, from this analysis, it appears that changing the values of `n_neighbors` would not decrease the gap between the training and testing accuracy doesn't decrease below 1% as it did for logistic regression. Therefore, including KNN in the grid search pipeline to fine the best model would not be a efficient use of computational time.

---

## Model 3: Decision Tree

### Initial Fitting

Let's fit a Decision Tree model to our data once using default hyper-parameter settings to gauge a baseline training and testing accuracy and precision metrics. We will use the same train-test split with `test_size` = 25% as was used for the logistic regression model, as well as the feature columns scaled using `StandardScalar`.

In [ ]:
model_obj = DecisionTreeClassifier
sm_dt = score_models(model_obj, model_param = {'min_samples_split':2}, model_name = 'DecisionTree)

In [ ]:
sm_dt.plt_scores();
plt.show()

With the default hyper-parameter settings (`max_depth`= None, `min_samples_split` = 2) the decision tree model was able to fit the training data with 97.35% accuracy and the testing data with 74.48% accuracy. While the decision tree model has the highest training accuracy of all the inital model fittings and second highest testing accuracy compared to the initial model fitting for logsitic regression (75.60%), the decision tree model also had the largest difference bewteen the training and testing accuracy (22.87%) between the three models. This difference suggests the decision tree model resulted in the most over fitting of the three models in their inital runs. However, like we suggested for the KNN modeling, the model accuracy and fitting can be improved by optimizing on the hyper-parameters. Regaurding the precision score, the 74.16% from the inital decision tree model suggest that the model is about classify the the observation correctly 74% of the time, which is improved from the inital precision of the KNN mdodel (68%) but slightly below the precision of the inital logistic regression model (76%).

### Optimizing Model Accuracy

To access how the model accuracy for decision trees changes with hyper-parameter optimization, let's observe how the model accuracy changes with integer step-sizes of `max_depth`. Here, the `max_depth` parameter will control for the number of nodes the decision tree can produce, which could result in the model over fitting to the trianing set if left too high. In its default state, the `max_depth` was set to `None` which left the termination to the node production to the defualt `min_samples_split` parameter, which defined how many observations were required to be categorized in a node for it to split.

In [ ]:
md_list = np.arange(5, 43, 2)
sm_knn.search('max_depth', md_list)

In [ ]:
fig = sm_knn.plt_hp_opt();
ax, = fig.get_axes();
ax.set_ylim([0.7, 1]);
plt.show()

From these results, we found that decision tree model achieved the least amount of over/under fitting (i.e., the most generalizability) for values of `max_depth` $< 9$, and over fitting starting to dominate the trend for values `max_depth` $\geq 9$. Interestingly, for the `max_depth` values between 5 and 9, both the training and testing acuracy rose from 73-77% while remaining the same. These findings suggests that the within those ranges of `max_depth`, the accuracy of the model can increase while maintaining generalizability. Alternatively, let's investigate if optimizing the `min_sample_split` instead of the `max_depth` parameter could lead to improved model accuracy.

In [ ]:
ss_list = np.arange(2, 42, 2)
sm_knn.search('min_samples_split', ss_list)

In [ ]:
fig = sm_knn.plt_hp_opt();
ax, = fig.get_axes();
ax.set_ylim([0.7, 1]);
plt.show()

From these results, we see that optimizing the `min_sample_split` parameter leads to more over-fitting of the model compared to the `max_depth` parameter. Therefore, the training and testing accuracy from the range of `max_depth` values between 5 and 9 demonstrate the `max_depth` parameter provides the best model generalizability. Regarding the classification accuracy, optimizing both the `min_sample_split` parameter and `max_depth` produce training accuracy > 85%, and testing accuracies near 75%. Therefore, since the parameter provides the results with the most generalizability, when optimizing the decision tree model using grid search, the `max_depth` parameter should be optimized between values of 5 and 9.

##  Model 4: Random Forest Model

### Initial Fitting

From the success of the decision tree model, let try to fit a random forest model to see if bootstrapping and aggregating will improve the accuracy of the decision tree model. 

Like the other models, the initial fit will be done using the default hyper-parameter settings to gauge a baseline training and testing accuracy and precision metrics. In this case, `n_estimators`  = 100. However, given the success of the decision tree model with a defined `max_depth` parameter between 5 and 9, will also define `max_depth` = 7 so that we can focus our study on optimizing the `n_estimators` parameter to achieve the best model accuracy. We will use the same train-test split with `test_size` = 25% as was used for the logistic regression model, as well as the scaled feature columns scaled using `StandardScalar`.

In [ ]:
model_obj = RandomForestClassifier
sm_rf = score_models(model_obj, model_param = {'n_estimators': 100, 'max_depth':7}, model_name = 'RandomForest')

In [ ]:
sm_rf.plt_scores();
plt.show()

With the default hyper-parameter settings (`n_estimators` = 100) and `max_depth` = 7,  the decision tree model could fit the training data with 78.78% accuracy and the testing data with 77.57% accuracy. With the difference between the training and testing scores being near 1% (1.21%), the generalizability of the initial fitting of the random forest model proves to be the most competitive with the initial fitting of the logistic regression model that had a training and testing score difference of 0.52%. In addition to their similar values for the difference between the training and testing sets, the training and testing accuracy and precision score for the initial random forest model (78.78%, 77.57%, 79.64%) is shown to be within 3% of the training and testing accuracy and precision score for the initial logistic regression model (76.12%, 74.60%, 74.31%), respectively. These results suggest that the random forest model could be the strongest competitor against logistic regression for classifying the citations into publication tiers compared to the initial modeling results alone. However, let's first investigate if optimizing the parameter `n_estimators` could potentially provide higher accuracies and improved generalizability compared to previously tested models.

### Optimizing Model Accuracy

Let's access how the model accuracy for the random forest model changes with optimization using large integer step-sizes of `n_estimators`. Here `n_estimators` determines the number of *trees* each 

In [ ]:
es_list = np.arange(10, 1000, 100)
sm_rf.search('n_estimators', es_list)

In [ ]:
fig = sm_rf.plt_hp_opt();
ax, = fig.get_axes();
ax.set_ylim([0.75, 0.80]);
ax.set_xscale('log');
plt.show()

From these results, we found that the training accuracy stays within 0.25% of 78.75%, and the testing accuracy stays within 0.7% of 77.50% for the range of estimators. These results suggest that the training and testing accuracy is not sensitive to the `n_estimator` parameter. Additionally, the difference between the training and testing accuracy stays within 1.3-1.5%, suggesting that the model's generalizability is not sensitive to the `n_estimator` parameter. Therefore, when considering building a computationally efficient pipeline to find the best model, it would be most computationally efficient to optimize the random forest model without performing a grid search on `n_estimators` and instead using a value from this study that gives the smallest difference between the training and testing data, which in this case is `n_estimators` = 410. Here, computational time for the pipe line could be focused on fitting the the random forest model to an optimal value of `max_depth`, which suggested to be a critical parameter to optimize to ensure generalizability of individual decision trees.

## Choosing the Best Models for Grid Search

To determine the best models and parameters for the grid search, let's compare the accuracy and precision results of the initial fitting predictions for each model with their pseudo-optimal hyper-parameters to determine which models have the most potential to classify the citations into tiers. Here, the pseudo-optimal hyper-parameters will be chosen within the proposed range of values suggested by the hyper-parameter optimization. Though the fit fitted to the pseudo-optimal hyper-parameters won't represent the final model fit, it will be used to judge the potential of each individual model against each other.

In [ ]:
#Model order
model_list = ['LogReg', 'KNN', 'DecisionTrees', 'RandomForest']

# Define inital model (default settings)
in_logit = LogisticRegression(solver='lbfgs', random_state=1)
in_knn = KNeighborsClassifier()
in_dt = DecisionTreeClassifier()
in_rf = RandomForestClassifier(max_depth = 7)

in_models = [in_logit, in_knn, in_dt, in_rf]

# Define pseudo-optimal models 
po_logit = LogisticRegression(solver='lbfgs', random_state=1, C = 10E-2)
po_knn = KNeighborsClassifier(n_neighbors = 33)
po_dt = DecisionTreeClassifier(max_depth = 7)
po_rf = RandomForestClassifier(n_estimators = 410, max_depth = 7)

po_models = [po_logit, po_knn, po_dt, po_rf]

In [ ]:
# Creating a dictionary to store the results in
dic = {'model': [], 'Training Score':[], 'Testing Score': [], 'Training/Test Score Difference': [], 'Weighted\nPrecision Score':[], 'Custom Score':[]}

In [ ]:
# for psuedo-optimized models

for i in range(len(po_models)):
    
    # 1. Grab Model
    model = po_models[i]
    
    if i == 0 and model_list[i] == 'LogReg': # if LogRegression, perform PCA transformation
        
        # Initiate PCA Model
        my_PCA = PCA(n_components = 30)

        # Fit to training data
        my_PCA.fit(X_train_scaled);
    
        X_train_PCA = my_PCA.transform(X_train_scaled)
        X_test_PCA = my_PCA.transform(X_test_scaled)
        
        # 2. Fit the Model
        model.fit(X_train_PCA, y_train)
        
        # 3. Score model
        train_score = model.score(X_train_PCA, y_train)
        test_score = model.score(X_test_PCA, y_test)
        diff = train_score-test_score

        # Get class predictions
        y_pred = model.predict(X_test_PCA)
        pre_score =  precision_score(y_test, y_pred, average='weighted')
        cs = custom_scorer(y_test, y_pred)
    
    else:
        # 2. Fit the Model
        model.fit(X_train_scaled, y_train)

        # 3. Score model
        train_score = model.score(X_train_scaled, y_train)
        test_score = model.score(X_test_scaled, y_test)
        diff = train_score-test_score

        # Get class predictions
        y_pred = model.predict(X_test_scaled)
        pre_score =  precision_score(y_test, y_pred, average='weighted')
        cs = custom_scorer(y_test, y_pred)

    dic['model'].append(model_list[i])
    dic['Training Score'].append(train_score)
    dic['Testing Score'].append(test_score)
    dic['Training/Test Score Difference'].append(diff)
    dic['Weighted\nPrecision Score'].append(pre_score)
    dic['Custom Score'].append(cs)

In [ ]:
# Initiate results dictionary
results_df = pd.DataFrame(dic)

In [ ]:
import matplotlib.gridspec as gridspec
color_pal = sns.color_palette("colorblind")
fig = plt.figure(figsize = (7.5, 2.5))

gs = fig.add_gridspec(nrows=1, ncols=2, width_ratios=[0.85, 0.15])
axes = [fig.add_subplot(gs[i]) for i in range(0,2)]

cols = ['Training Score', 'Testing Score', 'Weighted\nPrecision Score', 'Custom Score', 'Training/Test Score Difference']

j = 0
k = 0
for col in cols:
    i = 0
    for model in model_list:
        bar_val = results_df[results_df['model'] == model][col]
        if col != 'Training/Test Score Difference':
            axes[0].bar(j, bar_val, width = 0.25, color = color_pal[i], label = model)
            j += 0.25
            i += 1
        elif col == 'Training/Test Score Difference':
            axes[1].bar(k, bar_val, width = 1, color = color_pal[k], label = model)
            k += 1
    j += 0.25

# for axes [0,0]
axes[1].legend(title = 'Method', bbox_to_anchor=(1.05, 1.025))

# for axes[0]
x_ticks = np.arange(0.375, 4.375, 1.25)
axes[0].set_xticks(x_ticks, cols[:4]);
y_ticks = np.arange(0, 1, 0.05)
y_labels = ['{:.0%}'.format(i) for i in y_ticks]
axes[0].set_yticks(y_ticks, y_labels)
axes[0].set_ylim([0.70, 0.85])
axes[0].set_title('a)', loc = 'left')

# for axes[1]
x_ticks = [1.5]
y_ticks = np.arange(0, 1, 0.005)
y_labels = ['0%'] + ['{:.1%}'.format(i) for i in y_ticks[1:]]
axes[1].set_yticks(y_ticks, y_labels)
axes[1].set_xticks(ticks = x_ticks, labels = [cols[-1]]);
axes[1].set_ylim([-0.001, 0.02])
axes[1].set_title('b)', loc = 'left')

fig.tight_layout()

From the summary of the pseudo-optimized model scores, we observe that the random forest model has the highest training and testing accuracy and weighted precision scores, but the second largest difference between training and testing scores. Therefore, the summary suggests that using the random forest model could provide the best scores, but does not provide the best generalizability. In contrast, the pseudo-optimized decision tree model demonstrated having the lowest difference between training and testing scores, but had the second highest training, testing and precision scores. Additionally, the pseudo-optimized logistic regression model showed the third highest training, testing and precision scores and third smallest absolute difference between training and testing scores. These results suggest the decision tree model provides the best generalizability while also providing strong potential training, testing and precision scores, while the pseudo-optimized logistic regression model provides the third strongest performance overall. Finally, KNN demonstrated the worst pseudo-optimized model performance with the lowest training, testing and precision scores and highest absolute difference between training and testing scores.

Based on these results, we will implement the random forest model and decision tree model into the grid search pipeline to find the best parameters for classifying the citations into publication tiers. Since the best model will be determined based off of the average score between testing and precision scores, the random forest model will likely be chosen to be the best model. However, optimized values of `max_depth`, the decision tree model could overtake the random forest model in these categories.

In [ ]:
from tempfile import mkdtemp
# Define features and target colums
X = ml_df.drop('PubTier', axis = 1)
y = ml_df['PubTier']

# Split the data set
X_remainder, X_test, y_remainder, y_test =  train_test_split(X, y, test_size=0.25, random_state = 1)

#Define estimators
estimators = [
    ('scalar', StandardScaler()),
    ('dim', PCA())
    ('model', DecisionTreeClassifier())]

depth_vals = np.arange(5, 10, 1)
n_est_vals = np.arange(408, 412, 1)

# Initiate Pipeline object
cachedir = mkdtemp(dir = r'D:\Git\tmp')
pipe = Pipeline(estimators, memory = cachedir)

# Define parameter grid for grid search
param_grid = [
    {'model': [RandomForestClassifier()],
     'scalar':[StandardScaler()],
     'model__n_estimators': n_est_vals,
    'model__max_depth': depth_vals},
    {'model': [DecisionTreeClassifier()],
     'scalar':[StandardScaler()],
    'model__max_depth': depth_vals}
]

# Initiate the GridSearchCV object with a 5-fold cross validation
grid = GridSearchCV(pipe, param_grid=param_grid, cv = 5, scoring=make_scorer(custom_scorer))

# Perform the grid search, fitting the grid search to the training data only
grid.fit(X_remainder, y_remainder)

In [ ]:
import joblib

#save your model or results
joblib.dump(grid, 'NB5_gsModel.pkl')